In [ ]:
import fastf1
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
fastf1.Cache.enable_cache('../cache')
print("Libraries imported successfully!")

In [ ]:
schedule_2025 = fastf1.get_event_schedule(2025)
print("2025 F1 Season Calendar:")
print(schedule_2025[['RoundNumber', 'EventName', 'EventDate', 'Country']])

In [ ]:
session = fastf1.get_session(2025, 'Monaco', 'R')
session.load()

session = fastf1.get_session(2025, 'Abu Dhabi', 'R')
session.load()

session = fastf1.get_session(2025, 'Monza', 'R')
session.load()


In [ ]:
session = fastf1.get_session(2025, 'Monza', 'R')
session.load()

print(f"✅ Analyzing: {session.event['EventName']} - {session.name}")
print(f"📊 Total laps in session: {len(session.laps)}")

In [ ]:
laps = session.laps

print("Available columns:")
print(laps.columns.tolist())

print("\nFirst 5 laps:")
print(laps.head())

print("\nBasic stats:")
print(laps.describe())

In [ ]:
print(f"Starting with {len(laps)} laps")

clean_laps = laps[laps['LapTime'].notna()].copy()
print(f"After removing NaN times: {len(clean_laps)} laps")

clean_laps['LapTimeSeconds'] = clean_laps['LapTime'].dt.total_seconds()

fastest_lap = clean_laps['LapTimeSeconds'].min()
clean_laps = clean_laps[clean_laps['LapTimeSeconds'] < (fastest_lap + 30)]
print(f"After removing outliers: {len(clean_laps)} laps")

if 'PitInTime' in clean_laps.columns:
    clean_laps = clean_laps[clean_laps['PitInTime'].isna()]
    print(f"After removing pit laps: {len(clean_laps)} laps")

print(f"\n✅ Clean data ready: {len(clean_laps)} valid laps")

In [ ]:
compound_stats = clean_laps.groupby('Compound').agg({
    'LapTimeSeconds': ['mean', 'median', 'min', 'count']
}).round(2)

print("🏎️ Lap Time Analysis by Tire Compound")
print("=" * 60)
print(compound_stats)

# Mean gaps
compounds_mean = compound_stats['LapTimeSeconds']['mean'].sort_values()
print(f"\n📊 Performance Gaps (mean):")
if len(compounds_mean) >= 2:
    for i in range(len(compounds_mean) - 1):
        c1 = compounds_mean.index[i]
        c2 = compounds_mean.index[i + 1]
        gap = compounds_mean[c2] - compounds_mean[c1]
        print(f"{c1} vs {c2}: {gap:.2f}s faster")

# Median gaps
compounds_median = compound_stats['LapTimeSeconds']['median'].sort_values()
print(f"\n📊 Performance Gaps (median):")
if len(compounds_median) >= 2:
    for i in range(len(compounds_median) - 1):
        c1 = compounds_median.index[i]
        c2 = compounds_median.index[i + 1]
        gap = compounds_median[c2] - compounds_median[c1]
        print(f"{c1} vs {c2}: {gap:.2f}s faster")
# Min gaps
compounds_min = compound_stats['LapTimeSeconds']['min'].sort_values()
print(f"\n📊 Performance Gaps (min):")
if len(compounds_min) >= 2:
    for i in range(len(compounds_min) - 1):
        c1 = compounds_min.index[i]
        c2 = compounds_min.index[i + 1]
        gap = compounds_min[c2] - compounds_min[c1]
        print(f"{c1} vs {c2}: {gap:.2f}s faster")

#count gaps
compounds_count = compound_stats['LapTimeSeconds']['count'].sort_values()
print(f"\n📊 Lap Count by Compound:")
if len(compounds_count) >= 2:
    for i in range(len(compounds_count)):
        c = compounds_count.index[i]
        count = compounds_count[c]
        print(f"{c}: {count} laps")
print(compounds_count)

In [ ]:
#Mean Lap Time Chart
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 6))

compound_avg = clean_laps.groupby('Compound')['LapTimeSeconds'].mean().sort_values()

#Create Bar Chart
bars = ax.bar(compound_avg.index, compound_avg.values, 
               color=['#F0F0F0', '#FFF200', "#FF1E1E"],  # F1 tire colors
               edgecolor='black', linewidth=1.5)
#add Value Labels to Bar Chart
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}s',
            ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Average Lap Time (seconds)', fontsize=12, fontweight='bold')
ax.set_xlabel('Tire Compound', fontsize=12, fontweight='bold')
ax.set_title(f'2025 Monza GP - Average Lap Time by Tire Compound', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/monza_tire_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Chart saved to outputs/monza_tire_comparison.png")


In [ ]:
# Median Lap Time Chart
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 6))

compound_median = clean_laps.groupby('Compound')['LapTimeSeconds'].median().sort_values()
bars = ax.bar(compound_median.index, compound_median.values, 
               color = ['#FF1E1E', '#F0F0F0', '#FFF200'], # F1 tire colors
               edgecolor='black', linewidth=1.5)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}s',
            ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Median Lap Time (seconds)', fontsize=12, fontweight='bold')
ax.set_xlabel('Tire Compound', fontsize=12, fontweight='bold')
ax.set_title('2025 Monza GP - Median Lap Time by Tire Compound', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/monza_tire_median.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Chart saved to outputs/monza_tire_median.png")


In [ ]:
# Chart 2: Fastest Lap Time by Compound
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 6))

compound_min = clean_laps.groupby('Compound')['LapTimeSeconds'].min().sort_values()

# Create Bar Chart
bars = ax.bar(compound_min.index, compound_min.values, 
               color=['#FF1E1E', '#F0F0F0', '#FFF200'],  # Colors match sorted order
               edgecolor='black', linewidth=1.5)

# Add Value Labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}s',
            ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Fastest Lap Time (seconds)', fontsize=12, fontweight='bold')
ax.set_xlabel('Tire Compound', fontsize=12, fontweight='bold')
ax.set_title('2025 Monza GP - Fastest Lap by Tire Compound', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/monza_tire_fastest.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Chart saved to outputs/monza_tire_fastest.png")

In [ ]:
# Chart 3: Number of Laps by Compound
fig, ax = plt.subplots(figsize=(10, 6))

compound_count = clean_laps.groupby('Compound')['LapTimeSeconds'].count().sort_values()

# Create Bar Chart
bars = ax.bar(compound_count.index, compound_count.values, 
               color=['#FF1E1E', '#FFF200', '#F0F0F0'],  # Colors match sorted order
               edgecolor='black', linewidth=1.5)

# Add Value Labels
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{int(height)} laps',  # Use int for count, not decimals
            ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Number of Laps', fontsize=12, fontweight='bold')
ax.set_xlabel('Tire Compound', fontsize=12, fontweight='bold')
ax.set_title('2025 Monza GP - Lap Count by Tire Compound', 
             fontsize=14, fontweight='bold', pad=20)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/monza_tire_count.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Chart saved to outputs/monza_tire_count.png")